## 测试用例维护难？LangSmith自动化测试实现持续验证


### 测试维护的三大痛点：为什么手动测试是"性能刺客"

#### 痛点一：手动执行成本疯狂燃烧

当测试用例数量增长到一定规模时，人工执行的方式迅速变得不可持续。

#### 痛点二：版本管理混乱，回归测试成噩梦

大模型应用的核心组件——如 prompt、模型选择、参数配置等——经常变动。然而，传统测试方法缺乏对这些变更的有效追踪机制。

**版本追踪困难：**
```shell
v1.0: GPT-3.5 + 基础 prompt
v1.1: GPT-3.5 + 优化 prompt（提升准确率 5%）
v1.2: GPT-4 + 优化 prompt（成本上升 3 倍）
v1.3: GPT-3.5 + 新 prompt（回退方案）
```
核心问题：如何确认 v1.3 的表现不低于 v1.1？

手动测试无法提供稳定、可量化的跨版本对比数据，导致决策缺乏依据。

**测试用例与应用状态脱节**
- 应用已升级至 v2.0，但测试集仍基于旧版 prompt 设计；
- 新增功能未及时补充测试用例，造成覆盖盲区；
- 历史测试样本与当前业务场景不再匹配，失去参考价值。

**回归风险难以发现**
缺乏自动化的回归测试流程，常出现“修复一个问题，引入另一个问题”的情况。
例如：一次 prompt 调整优化了单轮问答准确性，却破坏了多轮对话的状态维持能力。由于没有对应的回归测试用例，该问题直到用户反馈才被发现。

#### 痛点三：测试结果不可信，质量门禁形同虚设

**评估标准不一致**

以测试用例“总结这篇财经新闻”为例：
- 期望输出：简洁、准确的摘要
- 实际输出：一段 200 字的详细总结

不同角色可能做出不同判断：
- 工程师 A：内容完整准确 → 通过
- 工程师 B：过于冗长，不符合“简洁”要求 → 失败
- 产品经理：用户偏好详细信息 → 接受
这种分歧反映出测试标准的模糊性。

**缺乏量化指标**
- “准确性”、“相关性”、“完整性”等维度缺乏可计算的度量方式；
- 测试通过率统计依赖人工打分，误差大；
- 性能退化趋势无法被系统性识别。

**质量门禁形同虚设**
由于测试过程不可控、结果不可信，团队往往倾向于降低准入门槛，甚至跳过测试直接上线。这形成了恶性循环：

```shell
测试不可靠 → 放宽标准 → 线上问题增多 → 更不信任测试 → 进一步弱化测试
```

## LangSmith自动化测试核心机制：Dataset + Run + Feedback 三位一体

LangSmith 的核心突破是构建了一套完整的"测试-执行-评估"闭环体系。简单说就是：**把原本需要人工干预的每个环节都自动化了**。不同于传统测试工具的单点解决方案，LangSmith 通过三个核心组件的协同工作，实现了大模型应用测试的全流程自动化。

### 核心架构：三层解耦设计

LangSmith 提供了一套面向大模型应用的系统性测试框架，通过 Dataset（数据集）、Run（执行记录） 和 Feedback（评估反馈） 三个核心组件的协同，构建了一个完整的“准备—执行—评估”闭环。

```shell
┌─────────────────┐    ┌─────────────────┐    ┌─────────────────┐
│   Dataset       │───▶│      Run        │───▶│   Feedback      │
│   (测试数据层)   │    │   (执行记录层)   │    │   (评估结果层)   │
└─────────────────┘    └─────────────────┘    └─────────────────┘
│                      │                      │
│ • 测试用例集合        │ • 执行轨迹记录        │ • 多维度评估指标
│ • 输入输出样本        │ • 性能监控数据        │ • 自动化评分规则  
│ • 版本管理机制        │ • 错误异常捕获        │ • 人工反馈收集
```

这种设计的核心价值在于：

- 测试数据与执行逻辑解耦 → Dataset 可被多个版本的应用重复使用
- 执行过程全程可观测 → 每个 Run 都是完整的 Trace，支持深度调试
- 评估标准结构化 → Feedback 支持多维度指标聚合，便于横向对比

就像微服务架构中的职责划分一样，每一层专注解决一类问题，整体形成高内聚、低耦合的测试体系。

### Dataset：不只是测试用例，更是知识资产

1.  多种创建方式，适配不同场景
（1）手动创建基础测试集

```python
from langsmith import Client

client = Client()

# 创建一个智能客服问答测试集
dataset = client.create_dataset(
    dataset_name="customer_service_qa",
    description="用于验证客服机器人回答准确性的测试集"
)
```
（2）从生产环境导入高质量样本

```python
# 从已有的 trace run 中提取表现良好的调用作为测试样例
client.create_examples_from_runs(
    run_ids=production_run_ids,           # 生产环境中采集的 run ID 列表
    dataset_id=dataset.id,
    source_run_map={r: r for r in production_run_ids},  # 映射原始 run
    include_outputs=True,                # 包含实际输出作为 reference
    filter_fn=lambda run: run.feedback_stats.get("positive", 0) >= 1  # 过滤出有正向反馈的调用
)
```

（3）批量导入历史测试数据

```python
import pandas as pd

test_cases = pd.read_csv("legacy_test_cases.csv")

for _, row in test_cases.iterrows():
    client.create_example(
        dataset_id=dataset.id,
        inputs={"question": row["input"]},
        outputs={"answer": row["expected_output"]},  # reference output
        metadata={
            "category": row["type"],
            "difficulty": row["level"],
            "source": "migration-2024"
        }
    )
```

2. 支持基于种子数据扩展
LangSmith 本身不提供“自动扩增测试用例”的功能，但你可以结合 LLM 实现数据增强逻辑。例如：

```python
def expand_test_cases(seed_examples, llm, target_count=200):
    """
    使用 LLM 基于种子样例生成语义相似的新测试用例
    """
    expanded = []
    for example in seed_examples:
        prompt = f"""
        请生成5个与以下问题语义相似但表述不同的变体：
        
        原始问题：{example['inputs']['question']}
        要求：
        1. 保持问题类型一致（如都是退换货咨询）
        2. 改变措辞、语气或细节描述
        3. 输出格式为每行一个问题
        """
        response = llm.invoke(prompt)
        questions = [q.strip() for q in response.split("\n") if q.strip()]
        
        for q in questions:
            expanded.append({
                "inputs": {"question": q},
                "outputs": example["outputs"],  # 共享参考答案
                "metadata": {**example.get("metadata", {}), "generated": True}
            })
    
    return expanded[:target_count]
```

3. 版本化管理：保障测试与应用同步演进

```python
{
  "dataset_id": "ds-customer-service-v2.1",
  "version": "2.1.0",
  "created_at": "2024-03-15T10:00:00Z",
  "metadata": {
    "model_version": "gpt-4-turbo",
    "prompt_version": "v2.1",
    "test_scenarios": ["基础问答", "多轮对话", "异常处理"],
    "coverage_rate": 0.95
  },
  "parent_dataset": "ds-customer-service-v2.0"
}
```

### Run：全链路执行记录，实现端到端可观测性

1. Trace 结构：还原完整的执行路径

每一次测试执行都会生成一个 DAG（有向无环图）结构的 trace，详细记录各组件的调用顺序和耗时：

```python
{
  "run_id": "run-test-20240315-001",
  "trace_group_id": "exp-regression-v2.1",
  "dataset_id": "ds-customer-service-v2.1",
  "run_type": "chain",
  "name": "customer_service_chain",
  "start_time": "2024-03-15T10:30:00Z",
  "end_time": "2024-03-15T10:30:02.5Z",
  "inputs": {"question": "如何退换货？"},
  "outputs": {"answer": "您可以在订单页面申请退换货..."},
  "child_runs": [
    {
      "run_id": "run-retrieval-001",
      "name": "knowledge_retrieval",
      "run_type": "retriever",
      "inputs": {"query": "退换货流程"},
      "outputs": {"documents": [...]},
      "execution_order": 1
    },
    {
      "run_id": "run-llm-001",
      "name": "answer_generation",
      "run_type": "llm",
      "inputs": {"context": "...", "question": "..."},
      "outputs": {"text": "您可以在订单页面..."},
      "execution_order": 2,
      "metrics": {
        "token_usage": {
          "input_tokens": 800,
          "output_tokens": 120,
          "total_tokens": 920
        },
        "cost_usd": 0.0021
      }
    }
  ],
  "tags": ["v2.1", "regression_test", "prod_candidate"],
  "metadata": {
    "latency_ms": 2500
  }
}
```

2. 批量执行测试套件

LangSmith 支持对整个 Dataset 进行批量运行，并自动记录每个 Example 的执行结果：
```python
from langsmith.evaluation import run_on_dataset
from langchain_core.runnables import RunnableConfig

def run_test_suite(dataset_name: str, chain_factory):
    """运行指定数据集上的测试套件"""
    results = run_on_dataset(
        client=client,
        dataset_name=dataset_name,
        llm_or_chain_factory=chain_factory,
        evaluation=None,  # 可选：嵌入评估器
        project_name=f"test-run-{datetime.now().strftime('%Y%m%d-%H%M')}",
        verbose=True,
        # 并发控制（推荐 5~10，避免触发 API 限流）
        config=RunnableConfig(max_concurrency=5)
    )
    return results
```



## Feedback：建立客观、可量化的质量评估体系

Feedback 是 LangSmith 中用于记录评估结果的核心对象。它可以表示自动化评分、人工打分或用户反馈，使质量判断从主观走向客观。

1. 自动化评估器示例
LangSmith 支持集成自定义评估逻辑。以下是几个常见维度的实现方式：

```python
from langsmith.schemas import EvaluationResult
from langsmith.evaluation import evaluate

# 定义相关性评估函数
def relevance_evaluator(run, example):
    prediction = run.outputs["answer"]
    question = run.inputs["question"]
    context = example.metadata.get("context", "")

    score = llm_judge.invoke(
        f"问题：{question}\n答案：{prediction}\n上下文：{context}\n"
        "请判断答案是否直接回应了问题（0=完全无关，1=完全相关）"
    )
    try:
        value = float(score.strip())
    except ValueError:
        value = 0.0

    return EvaluationResult(key="relevance", score=value)

# 定义完整性评估
def completeness_evaluator(run, example):
    expected_answer = example.outputs["answer"]
    actual_answer = run.outputs["answer"]

    key_sentences = extract_key_sentences(expected_answer)
    matched = [s for s in key_sentences if s in actual_answer]
    score = len(matched) / len(key_sentences) if key_sentences else 0

    return EvaluationResult(key="completeness", score=score)

# 格式正确性检查
def format_check_evaluator(run, example):
    try:
        validate_json_schema(run.outputs["answer"], expected_schema)
        score = 1.0
    except ValidationError:
        score = 0.0
    return EvaluationResult(key="format_correctness", score=score)

# 在实验中同时运行多个评估器
evaluation_results = evaluate(
    chain_factory,
    data="customer_service_qa",
    evaluators=[
        relevance_evaluator,
        completeness_evaluator,
        format_check_evaluator
    ],
    experiment_prefix="qa-experiment-v2.1"
)

```
LangSmith 会为每个 Run 生成对应的 Feedback 记录，并在 UI 中可视化展示。

3. 聚合分析：生成结构化评估报告

LangSmith 自动汇总评估结果，支持导出或进一步分析：


```python
{
  "overall_metrics": {
    "relevance": {"avg": 0.91, "std": 0.08},
    "completeness": {"avg": 0.85, "std": 0.12},
    "format_correctness": {"avg": 0.96, "std": 0.04}
  },
  "performance": {
    "avg_latency_ms": 2100,
    "avg_cost_per_call_usd": 0.0023
  },
  "regression_comparison": {
    "baseline_experiment": "qa-experiment-v2.0",
    "relevance_change": "+0.03",
    "latency_change_ms": "-200"
  }
}
```


三者联动带来的关键能力：

1. 测试资产可持续积累
  - Dataset 成为企业知识资产的一部分
  - 历史 Run 构建性能基线
  - Feedback 数据反哺 prompt 优化


2. 质量评估可量化
  - 替代“感觉还行”的模糊判断
  - 支持多维度加权评分（如 (relevance*0.4 + completeness*0.4 + latency*0.2)）
  - 为 A/B 测试和灰度发布提供依据

3. 迭代过程可追溯
  - 每次变更的影响清晰可见
  - 支持精确回滚到任意历史状态
  - 问题定位从经验驱动转为数据驱动
  
这套机制的本质，是将大模型应用的测试从“手工作坊式”的零散操作，升级为“工业化流水线”式的系统工程。

## 完整实现方案：从零搭建自动化测试体系

### 环境配置：5分钟完成基础设施搭建

LangSmith 的集成非常轻量，只需几个关键步骤即可启用全链路追踪和评估能力。

步骤一：安装必要依赖

步骤二：配置环境变量

步骤三：验证连接与追踪功能

### 测试套件创建：构建可复用的测试资产

步骤一：定义标准化的测试用例结构

步骤二：创建 Dataset 并导入测试用例

步骤三：从生产环境采集高质量测试样本

### 自动化执行：批量测试与并行优化

步骤一：定义待测的应用链（Application Chain）

步骤二：执行批量测试实验（Experiment）

步骤三：结果分析与报告生成

### 回归测试策略：确保质量不退化

回归测试是生产环境质量保障的最后一道防线。我们需要建立系统性的回归测试机制，确保每次变更都不会破坏现有功能。



## 总结

通过本文的完整实践，我们构建了一套端到端的大模型应用自动化测试体系：以 Dataset 为测试资产载体，通过 Run 实现全链路可观测执行，再借助 Feedback 完成多维度量化评估，最终形成“数据—执行—反馈”的闭环。

这一体系不仅解决了传统手动测试成本高、标准模糊、难以复现等痛点，更支持版本对比、基线管理与持续集成，真正实现了质量保障的工程化。

LangSmith 并未取代工程师的判断，而是将主观经验转化为可积累的数据资产——每一个测试用例、每一次运行记录、每一条评估结果，都在为系统的可靠性添砖加瓦。

当变更来临时，我们不再依赖“感觉还好”，而是能打开控制台，指着一份实验报告说：“这个版本的相关性提升了 7%，延迟下降了 200ms，可以安全上线。”

自动化测试的本质，是从“救火式运维”走向“预防式质量建设”。而这，正是大模型应用迈向工业级交付的关键一步。